# Successful vs Unsuccessful Grant Comparison

Run the scoring pipeline on all PDFs in `data/successful/` and `data/unsuccessful/`, print results, and export to Excel.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault('OLLAMA_HOST', 'http://localhost:11435')
os.environ.setdefault('OLLAMA_NUM_CTX', '131072')

from qwen3_ollama import _Scorer, score_application
from src.all_type_parser.all_type_parser import parse_and_save
from src.pool.build_pool import build_chunk_pool
from src.scoring.pipeline import (
    OVERALL_EXCLUDED_SECTIONS_BY_DOC_TYPE,
    SECTION_EXCLUDED_SUB_IDS_BY_DOC_TYPE,
    _aggregate_overall,
    _aggregate_section,
    _build_scored_section,
    _generate_json_with_parse_retry,
    _normalize_model_section_output,
    load_rubric,
)

CRITERIA_PATH = PROJECT_ROOT / 'criteria_points.json'
SUCCESSFUL_DIR = PROJECT_ROOT / 'data' / 'successful'
UNSUCCESSFUL_DIR = PROJECT_ROOT / 'data' / 'unsuccessful'
RESULTS_DIR = PROJECT_ROOT / 'experiments' / 'results' / 'compare'
PARSED_CACHE_DIR = RESULTS_DIR / 'parsed_cache'
EVIDENCE_DIR = RESULTS_DIR / 'evidence'

EXPERIMENT_OLLAMA_MODEL = 'qwen3.5:27b'
BASELINE_MAX_TOKENS = int(os.environ.get('WHOLE_JSON_BASELINE_MAX_TOKENS', '65536'))

# Minimum character count of the full parsed JSON — PDFs below this are skipped
MIN_PARSED_CHARS = 2000

SECTION_KEYS = [
    'general',
    'proposed_research',
    'training_development',
    'sites_support',
    'wpcc',
    'application_form',
]

for path in [RESULTS_DIR, PARSED_CACHE_DIR, EVIDENCE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Project root      :', PROJECT_ROOT)
print('Results dir       :', RESULTS_DIR)
print('Ollama host       :', os.environ.get('OLLAMA_HOST'))
print('Ollama num_ctx    :', os.environ.get('OLLAMA_NUM_CTX'))
print('Model             :', EXPERIMENT_OLLAMA_MODEL)
print('Baseline max tok  :', BASELINE_MAX_TOKENS)
print('Min parsed chars  :', MIN_PARSED_CHARS)

In [ ]:
def parse_pdf_cached(pdf_path: Path, *, reparse: bool = False) -> tuple[dict, Path]:
    json_path = PARSED_CACHE_DIR / f'{pdf_path.stem}.json'
    if reparse or not json_path.exists():
        parse_and_save(str(pdf_path), str(json_path))
    raw = json_path.read_text(encoding='utf-8').strip()
    if not raw:
        raise ValueError(f'Parsed JSON is empty for {pdf_path.name}')
    if len(raw) < MIN_PARSED_CHARS:
        raise ValueError(
            f'Parsed JSON too short for {pdf_path.name}: {len(raw)} chars < {MIN_PARSED_CHARS} threshold'
        )
    parsed = json.loads(raw)
    return parsed, json_path


def score_pdf(pdf_path: Path, *, scorer: _Scorer, reparse: bool = False) -> dict:
    parsed, parsed_json_path = parse_pdf_cached(pdf_path, reparse=reparse)
    artifact_dir = RESULTS_DIR / 'artifacts' / pdf_path.stem
    artifact_dir.mkdir(parents=True, exist_ok=True)
    result = score_application(
        parsed, CRITERIA_PATH,
        doc_id=pdf_path.stem, scorer=scorer, artifacts_dir=artifact_dir,
    )
    return result


_BASELINE_SYSTEM_PROMPT = (
    'You are a strict NIHR grant scoring baseline. You do not use staged retrieval, '
    'belief state, or multi-pass reasoning. Return JSON only and follow the schema exactly.\n\n'
    'Score every signal in every criterion using the same strict 0-5 standard as the pipeline scorer. '
    'A signal score must be an integer from 0 to 5 inclusive. '
    'Scoring guide: 0=no evidence, 1=very weak, 2=weak, 3=moderate, '
    '4=strong with only minor gaps, 5=perfectly met: complete, specific, directly evidenced, '
    'and with no material caveats or missing detail. '
    'Do not give high scores merely because a topic is mentioned. Generic, aspirational, inferred, '
    'or weakly supported claims should usually score 2-3. '
    'If evidence is missing, unclear, or only partially supports the signal, give a low or moderate score, not 4-5. '
    'If any caveat, missing detail, weak support, or partial support applies to a signal, that signal must not receive 5.\n\n'
    'Identifier rules: each top-level key must be a parent sub_id. Inside signals, use full signal ids. '
    'Every signal belonging to a subcriterion must appear. used_chunk_ids must come only from evidence_chunk_index.\n\n'
    'OUTPUT FORMAT\n'
    '=============\n'
    'Return a single JSON object. Every section key and every sub-criterion key must be present. '
    'Example (two sub-criteria shown; your output must cover ALL criteria):\n\n'
    '{\n'
    '  "general": {\n'
    '    "g.1": {\n'
    '      "signals": {"g.1.a": 4, "g.1.b": 3, "g.1.c": 2, "g.1.d": 4},\n'
    '      "used_chunk_ids": ["chunk_003", "chunk_017"]\n'
    '    },\n'
    '    "g.2": {\n'
    '      "signals": {"g.2.a": 5, "g.2.b": 4, "g.2.c": 3},\n'
    '      "used_chunk_ids": ["chunk_001"]\n'
    '    }\n'
    '  }\n'
    '}'
)

def baseline_artifact_dir(pdf_path: Path) -> Path:
    return EVIDENCE_DIR / pdf_path.stem


def baseline_features_path(pdf_path: Path) -> Path:
    return baseline_artifact_dir(pdf_path) / f'{pdf_path.stem}_baseline_features.json'


def baseline_artifacts_complete(pdf_path: Path) -> bool:
    return baseline_features_path(pdf_path).exists()


def _strip_expanded_evidence_text(features: dict) -> dict:
    clean = json.loads(json.dumps(features))
    for section_data in clean.values():
        for sub in section_data.get('sub_criteria', []):
            sub.pop('evidence', None)
    return clean


def _flatten_baseline_subcriteria_evidence(features: dict) -> list[dict]:
    rows = []
    for section_key, section_data in features.items():
        for sub in section_data.get('sub_criteria', []):
            rows.append({
                'section_key': section_key,
                'sub_id': sub.get('sub_id'),
                'name': sub.get('name'),
                'score_10': sub.get('score_10'),
                'quality': sub.get('quality'),
                'used_chunk_ids': sub.get('used_chunk_ids', []),
                'evidence_count': sub.get('evidence_count', 0),
                'signals': sub.get('signals', []),
                'pros': sub.get('pros', ''),
                'drawbacks': sub.get('drawbacks', ''),
            })
    return rows


def _write_baseline_artifacts(
    *,
    pdf_path: Path,
    raw_response: str,
    parsed_response: dict,
    sections: list[dict],
    features: dict,
    overall: dict,
) -> dict[str, str]:
    artifact_dir = baseline_artifact_dir(pdf_path)
    artifact_dir.mkdir(parents=True, exist_ok=True)
    stem = pdf_path.stem

    paths = {
        'raw_response': artifact_dir / f'{stem}_baseline_raw_response.txt',
        'parsed_response': artifact_dir / f'{stem}_baseline_parsed_response.json',
        'scored_sections': artifact_dir / f'{stem}_baseline_scored_sections.json',
        'features': artifact_dir / f'{stem}_baseline_features.json',
        'subcriteria_evidence': artifact_dir / f'{stem}_baseline_subcriteria_evidence.json',
        'overall': artifact_dir / f'{stem}_baseline_overall.json',
    }

    paths['raw_response'].write_text(raw_response or '', encoding='utf-8')
    paths['parsed_response'].write_text(json.dumps(parsed_response, ensure_ascii=False, indent=2), encoding='utf-8')
    paths['scored_sections'].write_text(json.dumps(sections, ensure_ascii=False, indent=2), encoding='utf-8')
    paths['features'].write_text(json.dumps(_strip_expanded_evidence_text(features), ensure_ascii=False, indent=2), encoding='utf-8')
    paths['subcriteria_evidence'].write_text(
        json.dumps(_flatten_baseline_subcriteria_evidence(features), ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    paths['overall'].write_text(json.dumps(overall, ensure_ascii=False, indent=2), encoding='utf-8')
    return {key: str(value) for key, value in paths.items()}


def score_pdf_baseline(pdf_path: Path, *, scorer: _Scorer) -> dict:
    """One-shot baseline: full chunk index (untruncated text) + criteria, per-signal 0-5 scoring."""
    parsed, _ = parse_pdf_cached(pdf_path)
    doc_type = (parsed.get('doc_type') or '').lower()
    excluded_sections = OVERALL_EXCLUDED_SECTIONS_BY_DOC_TYPE.get(doc_type, set())
    excluded_sub_ids = SECTION_EXCLUDED_SUB_IDS_BY_DOC_TYPE.get(doc_type, set())

    rubric_sections = load_rubric(CRITERIA_PATH)
    pool_data = build_chunk_pool(parsed)
    pool_lookup = pool_data['pool_lookup']
    all_chunk_ids = list(pool_lookup)
    chunk_order = {cid: idx for idx, cid in enumerate(all_chunk_ids)}

    chunk_index = [
        {
            'chunk_id': cid,
            'parser_section': m.get('parser_section', ''),
            'source_path': m.get('source_path', ''),
            'text': m.get('text', ''),
        }
        for cid, m in pool_lookup.items()
    ]

    chunk_id_item = {'type': 'string', 'enum': all_chunk_ids} if all_chunk_ids else {'type': 'string'}
    sec_props = {}
    for sec in rubric_sections:
        sub_props = {}
        for sub in sec['sub_criteria']:
            signal_props = {sig['sid']: {'type': 'integer', 'enum': [0, 1, 2, 3, 4, 5]} for sig in sub['signals']}
            sub_props[sub['sub_id']] = {
                'type': 'object',
                'properties': {
                    'signals': {'type': 'object', 'properties': signal_props,
                                'required': list(signal_props), 'additionalProperties': False},
                    'used_chunk_ids': {'type': 'array', 'items': chunk_id_item, 'maxItems': 5},
                },
                'required': ['signals', 'used_chunk_ids'],
                'additionalProperties': False,
            }
        sec_props[sec['section_key']] = {
            'type': 'object', 'properties': sub_props,
            'required': list(sub_props), 'additionalProperties': False,
        }
    schema = {'type': 'object', 'properties': sec_props,
              'required': list(sec_props), 'additionalProperties': False}

    messages = [
        {'role': 'system', 'content': _BASELINE_SYSTEM_PROMPT},
        {
            'role': 'user',
            'content': json.dumps({
                'task': 'one_shot_chunk_index_baseline_scoring',
                'rules': [
                    'Use the criteria and evidence_chunk_index to score every signal.',
                    'Make exactly one direct score for every signal in every criterion.',
                    'Each signal score must be an integer from 0 to 5 inclusive.',
                    'Scoring guide: 0=no evidence, 1=very weak, 2=weak, 3=moderate, 4=strong with only minor gaps, 5=perfectly met: complete, specific, directly evidenced, and with no material caveats or missing detail.',
                    'Do not give high scores merely because a topic is mentioned; generic, aspirational, inferred, or weakly supported claims should usually score 2-3.',
                    'If evidence is missing, unclear, or only partially supports the signal, give a low or moderate score, not 4-5.',
                    'If any caveat, missing detail, weak support, or partial support applies to a signal, that signal must not receive 5.',
                    'Use used_chunk_ids only from evidence_chunk_index. If there is no relevant support, use an empty list and score weak or absent signals low.',
                ],
                'criteria': rubric_sections,
                'evidence_chunk_index': chunk_index,
            }, ensure_ascii=False),
        },
    ]

    raw_response, parsed_response, _ = _generate_json_with_parse_retry(
        scorer, messages, schema=schema, max_tokens=BASELINE_MAX_TOKENS, max_retries=0,
    )

    sections = []
    for rubric_section in rubric_sections:
        sk = rubric_section['section_key']
        raw_section = (parsed_response or {}).get(sk, {})
        if not isinstance(raw_section, dict):
            raw_section = {}
        normalized = _normalize_model_section_output(raw_section, rubric_section, all_chunk_ids)
        sections.append(_build_scored_section(rubric_section, normalized, chunk_order, excluded_sub_ids=excluded_sub_ids))

    features = {sec['section_key']: _aggregate_section(sec, pool_lookup) for sec in sections}
    section_weights = {sec['section_key']: sec['weight'] for sec in sections}
    overall = _aggregate_overall(features, section_weights, excluded_sections=excluded_sections)
    artifacts = _write_baseline_artifacts(
        pdf_path=pdf_path,
        raw_response=raw_response,
        parsed_response=parsed_response or {},
        sections=sections,
        features=features,
        overall=overall,
    )
    return {'overall': overall, 'features': features, 'artifacts': artifacts}


def get_overall(result: dict) -> float:
    return round(float(result.get('overall', {}).get('final_score_0to100', 0)), 2)


def build_progress_row(pdf_name: str, category: str, method: str, result: dict) -> dict:
    row = {
        'pdf_name': pdf_name,
        'category': category,
        f'{method}_overall_score_100': get_overall(result),
    }
    features = result.get('features', {})
    for section_key in SECTION_KEYS:
        section = features.get(section_key, {})
        overall = section.get('overall', {})
        row[f'{method}_{section_key}_score_100'] = overall.get('final_score_0to100')
        row[f'{method}_{section_key}_evidence_count'] = overall.get('evidence_count')
    return row


print('Helper functions defined.')

In [ ]:
# ── Run PIPELINE: SUCCESSFUL ──────────────────────────────────────────────────
PIPELINE_CSV_S = RESULTS_DIR / 'progress_pipeline_successful.csv'

scorer = _Scorer(model_name=EXPERIMENT_OLLAMA_MODEL)
successful_pdfs = sorted(SUCCESSFUL_DIR.glob('*.pdf'))
done_ps = set(pd.read_csv(PIPELINE_CSV_S)['pdf_name'].tolist()) if PIPELINE_CSV_S.exists() else set()
print(f'Pipeline successful: {len(done_ps)}/{len(successful_pdfs)} already done')

for idx, pdf_path in enumerate(successful_pdfs, start=1):
    if pdf_path.name in done_ps:
        print(f'[{idx}/{len(successful_pdfs)}] SKIP {pdf_path.name}'); continue
    print(f'\n[{idx}/{len(successful_pdfs)}] {pdf_path.name}')
    try:
        result = score_pdf(pdf_path, scorer=scorer)
        score = get_overall(result)
        row = build_progress_row(pdf_path.name, 'successful', 'pipeline', result)
        pd.DataFrame([row]).to_csv(PIPELINE_CSV_S, mode='a', header=not PIPELINE_CSV_S.exists(), index=False)
        done_ps.add(pdf_path.name)
        print(f'  pipeline={score:.1f}')
    except Exception as exc:
        print(f'  ERROR: {exc}')

print(f'\nDone. {len(done_ps)}/{len(successful_pdfs)} successful (pipeline)')

In [ ]:
# ── Run BASELINE: SUCCESSFUL ──────────────────────────────────────────────────
BASELINE_CSV_S = RESULTS_DIR / 'progress_baseline_successful.csv'

scorer = _Scorer(model_name=EXPERIMENT_OLLAMA_MODEL)
successful_pdfs = sorted(SUCCESSFUL_DIR.glob('*.pdf'))
done_bs = set(pd.read_csv(BASELINE_CSV_S)['pdf_name'].tolist()) if BASELINE_CSV_S.exists() else set()
print(f'Baseline successful: {len(done_bs)}/{len(successful_pdfs)} already done')

for idx, pdf_path in enumerate(successful_pdfs, start=1):
    csv_done = pdf_path.name in done_bs
    artifact_done = baseline_artifacts_complete(pdf_path)
    if csv_done and artifact_done:
        print(f'[{idx}/{len(successful_pdfs)}] SKIP {pdf_path.name}'); continue
    print(f'\n[{idx}/{len(successful_pdfs)}] {pdf_path.name}')
    try:
        result = score_pdf_baseline(pdf_path, scorer=scorer)
        score = get_overall(result)
        if not csv_done:
            row = build_progress_row(pdf_path.name, 'successful', 'baseline', result)
            pd.DataFrame([row]).to_csv(BASELINE_CSV_S, mode='a', header=not BASELINE_CSV_S.exists(), index=False)
            done_bs.add(pdf_path.name)
        print(f'  baseline={score:.1f}')
        print(f"  saved baseline evidence: {result['artifacts']['features']}")
    except Exception as exc:
        print(f'  ERROR: {exc}')

print(f'\nDone. {len(done_bs)}/{len(successful_pdfs)} successful (baseline)')

In [ ]:
# ── Run PIPELINE: UNSUCCESSFUL ────────────────────────────────────────────────
PIPELINE_CSV_U = RESULTS_DIR / 'progress_pipeline_unsuccessful.csv'

scorer = _Scorer(model_name=EXPERIMENT_OLLAMA_MODEL)
unsuccessful_pdfs = sorted(UNSUCCESSFUL_DIR.glob('*.pdf'))
done_pu = set(pd.read_csv(PIPELINE_CSV_U)['pdf_name'].tolist()) if PIPELINE_CSV_U.exists() else set()
print(f'Pipeline unsuccessful: {len(done_pu)}/{len(unsuccessful_pdfs)} already done')

for idx, pdf_path in enumerate(unsuccessful_pdfs, start=1):
    if pdf_path.name in done_pu:
        print(f'[{idx}/{len(unsuccessful_pdfs)}] SKIP {pdf_path.name}'); continue
    print(f'\n[{idx}/{len(unsuccessful_pdfs)}] {pdf_path.name}')
    try:
        result = score_pdf(pdf_path, scorer=scorer)
        score = get_overall(result)
        row = build_progress_row(pdf_path.name, 'unsuccessful', 'pipeline', result)
        pd.DataFrame([row]).to_csv(PIPELINE_CSV_U, mode='a', header=not PIPELINE_CSV_U.exists(), index=False)
        done_pu.add(pdf_path.name)
        print(f'  pipeline={score:.1f}')
    except Exception as exc:
        print(f'  ERROR: {exc}')

print(f'\nDone. {len(done_pu)}/{len(unsuccessful_pdfs)} unsuccessful (pipeline)')

In [ ]:
# ── Run BASELINE: UNSUCCESSFUL ────────────────────────────────────────────────
BASELINE_CSV_U = RESULTS_DIR / 'progress_baseline_unsuccessful.csv'

scorer = _Scorer(model_name=EXPERIMENT_OLLAMA_MODEL)
unsuccessful_pdfs = sorted(UNSUCCESSFUL_DIR.glob('*.pdf'))
done_bu = set(pd.read_csv(BASELINE_CSV_U)['pdf_name'].tolist()) if BASELINE_CSV_U.exists() else set()
print(f'Baseline unsuccessful: {len(done_bu)}/{len(unsuccessful_pdfs)} already done')

for idx, pdf_path in enumerate(unsuccessful_pdfs, start=1):
    csv_done = pdf_path.name in done_bu
    artifact_done = baseline_artifacts_complete(pdf_path)
    if csv_done and artifact_done:
        print(f'[{idx}/{len(unsuccessful_pdfs)}] SKIP {pdf_path.name}'); continue
    print(f'\n[{idx}/{len(unsuccessful_pdfs)}] {pdf_path.name}')
    try:
        result = score_pdf_baseline(pdf_path, scorer=scorer)
        score = get_overall(result)
        if not csv_done:
            row = build_progress_row(pdf_path.name, 'unsuccessful', 'baseline', result)
            pd.DataFrame([row]).to_csv(BASELINE_CSV_U, mode='a', header=not BASELINE_CSV_U.exists(), index=False)
            done_bu.add(pdf_path.name)
        print(f'  baseline={score:.1f}')
        print(f"  saved baseline evidence: {result['artifacts']['features']}")
    except Exception as exc:
        print(f'  ERROR: {exc}')

print(f'\nDone. {len(done_bu)}/{len(unsuccessful_pdfs)} unsuccessful (baseline)')

In [ ]:
# ── Analysis & Export ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np

# Merge all four CSVs
df_ps = pd.read_csv(PIPELINE_CSV_S)
df_bs = pd.read_csv(BASELINE_CSV_S).drop(columns=['category'], errors='ignore')
df_pu = pd.read_csv(PIPELINE_CSV_U)
df_bu = pd.read_csv(BASELINE_CSV_U).drop(columns=['category'], errors='ignore')

df_s = df_ps.merge(df_bs, on='pdf_name', how='outer')
df_u = df_pu.merge(df_bu, on='pdf_name', how='outer')
df = pd.concat([df_s, df_u], ignore_index=True).sort_values('pdf_name').reset_index(drop=True)

display(df[['pdf_name', 'category', 'pipeline_overall', 'baseline_overall']])

# ── Summary table ─────────────────────────────────────────────────────────────
summary = df.groupby('category')[['pipeline_overall', 'baseline_overall']].agg(['mean', 'std']).round(2)
print('\nGroup summary:')
display(summary)

# ── Box plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
fig.suptitle('Pipeline vs Baseline — Successful vs Unsuccessful', fontsize=13)

groups = ['successful', 'unsuccessful']
colors = ['#4C72B0', '#DD8452']

for ax_idx, (ax, method, col) in enumerate(zip(axes, ['Pipeline', 'Baseline'], ['pipeline_overall', 'baseline_overall'])):
    data = [df.loc[df['category'] == g, col].dropna().values for g in groups]
    bp = ax.boxplot(data, patch_artist=True, widths=0.5,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    for i, (d, color) in enumerate(zip(data, colors), start=1):
        ax.scatter(np.full(len(d), i) + np.random.uniform(-0.08, 0.08, len(d)),
                   d, color=color, s=30, zorder=3, alpha=0.8)

    means = [d.mean() if len(d) else float('nan') for d in data]
    ax.scatter([1, 2], means, marker='D', color='white', edgecolors='black', s=50, zorder=4)

    ax.set_title(method)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(groups)
    ax.set_ylabel('Score (0–100)')
    ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
    ax.grid(axis='y', alpha=0.3)

    for i, m in enumerate(means, start=1):
        if not np.isnan(m):
            ax.text(i, m + 1.5, f'{m:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    if ax_idx == 0:
        handles = [mpatches.Patch(facecolor=c, alpha=0.7, label=g) for c, g in zip(colors, groups)]
        ax.legend(handles=handles, loc='upper left', fontsize=9)

plt.tight_layout()
ts = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
fig_path = RESULTS_DIR / f'compare_{ts}.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

# ── Excel export ───────────────────────────────────────────────────────────────
excel_path = RESULTS_DIR / f'compare_{ts}.xlsx'
with pd.ExcelWriter(excel_path) as writer:
    df[['pdf_name', 'category', 'pipeline_overall', 'baseline_overall']].to_excel(
        writer, sheet_name='Scores', index=False)
    summary.to_excel(writer, sheet_name='Summary')
print(f'Exported: {excel_path}')

In [ ]:
# ── Distribution Comparison: Pipeline vs Baseline ─────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Load & merge all results ──────────────────────────────────────────────────
def load_merged(pipeline_csv: Path, baseline_csv: Path) -> pd.DataFrame:
    df_p = pd.read_csv(pipeline_csv)
    df_b = pd.read_csv(baseline_csv).drop(columns=['category'], errors='ignore')
    return df_p.merge(df_b, on='pdf_name', how='inner')

df_s = load_merged(PIPELINE_CSV_S, BASELINE_CSV_S).assign(group='successful')
df_u = load_merged(PIPELINE_CSV_U, BASELINE_CSV_U).assign(group='unsuccessful')
df_all = pd.concat([df_s, df_u], ignore_index=True)

score_keys = ['overall'] + SECTION_KEYS
required_score_cols = []
for method in ['pipeline', 'baseline']:
    required_score_cols.extend(
        [f'{method}_overall_score_100'] + [f'{method}_{key}_score_100' for key in SECTION_KEYS]
    )

# Keep only PDFs that have both pipeline and baseline scores for overall + all sections.
EXCLUDED_PARSING_CASES = {'IC00017.pdf', 'IC00061.pdf', 'IC00021.pdf', 'IC00094.pdf'}
GROUPS = ['successful', 'unsuccessful']
COLORS = {'successful': '#2ecc71', 'unsuccessful': '#e74c3c'}
ALPHA = 0.55
BINS = 10

df_complete = df_all.dropna(subset=required_score_cols).copy()
df_filtered = df_complete[~df_complete['pdf_name'].isin(EXCLUDED_PARSING_CASES)].copy()

# ── Excluded parsing-case table ───────────────────────────────────────────────
def build_excluded_case_table(frame: pd.DataFrame, excluded_cases: set[str]) -> pd.DataFrame:
    rows = []
    group_frames = [(group, frame[frame['group'] == group]) for group in GROUPS]
    group_frames.append(('overall', frame))

    for group, group_df in group_frames:
        excluded = group_df[group_df['pdf_name'].isin(excluded_cases)]
        rows.append({
            'group': group,
            'total_files': len(group_df),
            'excluded_files': len(excluded),
            'excluded_rate_%': round(len(excluded) / len(group_df) * 100, 1) if len(group_df) else float('nan'),
            'excluded_pdf_names': ', '.join(excluded['pdf_name'].sort_values()),
        })
    return pd.DataFrame(rows)

excluded_case_df = build_excluded_case_table(df_complete, EXCLUDED_PARSING_CASES)
print('\nExcluded failed parsing cases:')
print(', '.join(sorted(EXCLUDED_PARSING_CASES)))
display(excluded_case_df)

# ── Figure 1: Overall distributions only, pipeline and baseline side by side ──
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True)
fig.suptitle('Overall Score Distributions: Successful vs Unsuccessful',
             fontsize=13, fontweight='bold')

for ax, method, title in zip(axes, ['pipeline', 'baseline'], ['Pipeline (our method)', 'Baseline (one-shot)']):
    col_name = f'{method}_overall_score_100'
    bin_edges = np.linspace(0, 100, BINS + 1)

    for group in GROUPS:
        scores = df_complete.loc[df_complete['group'] == group, col_name].dropna()
        ax.hist(scores, bins=bin_edges, alpha=ALPHA, color=COLORS[group], label=group.title())
        if len(scores):
            ax.axvline(scores.mean(), color=COLORS[group], lw=2, ls='--')

    s_scores = df_complete.loc[df_complete['group'] == 'successful', col_name].dropna()
    u_scores = df_complete.loc[df_complete['group'] == 'unsuccessful', col_name].dropna()
    diff = s_scores.mean() - u_scores.mean()

    ax.set_title(f'{title}\nΔmean = {diff:+.1f}', fontsize=10)
    ax.set_xlabel('Score (0-100)', fontsize=9)
    ax.set_xlim(0, 100)
    ax.grid(axis='y', alpha=0.25)
    ax.legend(loc='upper left', fontsize=8)

axes[0].set_ylabel('Count', fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'distribution_overall.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: distribution_overall.png')

# ── Figures 2 & 3: Mean scores, complete set and parsing-filtered set ─────────
def plot_mean_scores(frame: pd.DataFrame, *, title: str, filename: str, figsize: tuple[float, float] = (17, 6.2)) -> pd.DataFrame:
    print(f'\nFiles included in {filename}:')
    display(frame[['group', 'pdf_name']].sort_values(['group', 'pdf_name']).reset_index(drop=True))

    # Determine y-axis floor: start just below the minimum value across all score columns
    all_score_cols = (
        [f'{m}_overall_score_100' for m in ['pipeline', 'baseline']]
        + [f'{m}_{k}_score_100' for m in ['pipeline', 'baseline'] for k in SECTION_KEYS]
    )
    min_val = frame[all_score_cols].min().min()
    y_floor = max(0, np.floor(min_val / 10) * 10 - 5)
    y_ceil = 107

    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=True)
    fig.suptitle(title, fontsize=13, fontweight='bold', y=0.98)

    x = np.arange(len(score_keys))
    tick_labels = ['Overall'] + [key.replace('_', '\n') for key in SECTION_KEYS]
    width = 0.35

    for ax, method in zip(axes, ['pipeline', 'baseline']):
        cols = [f'{method}_overall_score_100'] + [f'{method}_{key}_score_100' for key in SECTION_KEYS]
        means_by_group = {
            group: frame.loc[frame['group'] == group, cols].mean().values
            for group in GROUPS
        }

        ax.bar(x - width/2, means_by_group['successful'] - y_floor, width,
               bottom=y_floor, color=COLORS['successful'], alpha=0.8, label='Successful')
        ax.bar(x + width/2, means_by_group['unsuccessful'] - y_floor, width,
               bottom=y_floor, color=COLORS['unsuccessful'], alpha=0.8, label='Unsuccessful')

        ax.set_title('Pipeline (ours)' if method == 'pipeline' else 'Baseline (one-shot)',
                     fontsize=11, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(tick_labels, fontsize=9)
        ax.set_ylabel('Mean Score (0-100)')
        ax.set_ylim(y_floor, y_ceil)
        ax.grid(axis='y', alpha=0.3)
        ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), borderaxespad=0.0, fontsize=9)

        for i, (sm, um) in enumerate(zip(means_by_group['successful'], means_by_group['unsuccessful'])):
            if np.isfinite(sm) and np.isfinite(um):
                label_y = min(max(sm, um) + 1.5, y_ceil - 2)
                ax.text(x[i], label_y, f'Δ {sm - um:+.1f}',
                        ha='center', va='bottom', fontsize=8, color='#333333', clip_on=False)

    plt.tight_layout(rect=[0, 0, 0.92, 0.94])
    plt.savefig(RESULTS_DIR / filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {filename}')

    rows = []
    for method in ['pipeline', 'baseline']:
        for key in score_keys:
            col = f'{method}_overall_score_100' if key == 'overall' else f'{method}_{key}_score_100'
            s_m = frame.loc[frame['group'] == 'successful', col].mean()
            u_m = frame.loc[frame['group'] == 'unsuccessful', col].mean()
            rows.append({
                'method': method,
                'section': key,
                'successful_mean': round(s_m, 1),
                'unsuccessful_mean': round(u_m, 1),
                'delta': round(s_m - u_m, 1),
            })
    return pd.DataFrame(rows)

print(f'Complete matched PDFs: {len(df_complete)} / {len(df_all)}')
summary_complete = plot_mean_scores(
    df_complete,
    title='Mean Section Scores by Group - Complete Pipeline and Baseline Results',
    filename='mean_scores_complete.png',
)

print(f'After filtering excluded parsing cases: {len(df_filtered)} / {len(df_complete)}')
summary_filtered = plot_mean_scores(
    df_filtered,
    title='Mean Section Scores by Group - Filtered Failed Parsing Cases',
    filename='mean_scores_filtered_excluded_parsing_cases.png',
    figsize=(18.5, 5.4),
)

# ── Print summary tables ──────────────────────────────────────────────────────
print('\n── Complete matched PDFs: mean scores (successful vs unsuccessful) ──')
display(summary_complete.pivot(index='section', columns='method', values=['successful_mean', 'unsuccessful_mean', 'delta']))

print('\n── Filtered failed parsing cases: mean scores (successful vs unsuccessful) ──')
display(summary_filtered.pivot(index='section', columns='method', values=['successful_mean', 'unsuccessful_mean', 'delta']))